# KAEM Profiling

Set runtime to **Python** + **GPU** (T4) or **TPU** (v5e), then run all.
Results save to `profiling_reports/<device>/` and `figures/profiling/<device>/`.

In [ ]:
%%shell
set -e
if ! command -v julia &> /dev/null; then
    rm -f /root/.julia/juliaup/juliaup.json
    curl -fsSL https://install.julialang.org | sh -s -- -y --default-channel 1.11
fi
export PATH="$HOME/.juliaup/bin:$PATH"
julia --version
[ ! -d /content/KAEM ] && git clone https://github.com/PritRaj1/KAEM.git /content/KAEM
cd /content/KAEM
julia --project=. -e 'using Pkg; Pkg.instantiate()'

In [ ]:
import subprocess, os, json
os.environ["PATH"] = os.path.expanduser("~/.juliaup/bin") + ":" + os.environ["PATH"]
KAEM = "/content/KAEM"

r = subprocess.run("nvidia-smi", capture_output=True)
if r.returncode == 0:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()
    DEVICE, DEVICE_NAME = "gpu", gpu_name
elif os.path.exists("/dev/accel0"):
    DEVICE, DEVICE_NAME = "tpu", "TPU v5e-1"
else:
    DEVICE, DEVICE_NAME = "cpu", "CPU"
print(f"Device: {DEVICE} ({DEVICE_NAME})")

REPORT_DIR = f"{KAEM}/profiling_reports/{DEVICE}"
FIG_DIR = f"{KAEM}/figures/profiling/{DEVICE}"
TRACE_DIR = f"{REPORT_DIR}/traces"
os.makedirs(TRACE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

with open(f"{REPORT_DIR}/device_info.txt", "w") as f:
    f.write(f"device: {DEVICE}\ndevice_name: {DEVICE_NAME}\n")

def jl(code, timeout=3600):
    r = subprocess.run(["julia", "--project=.", "-e", code],
                       cwd=KAEM, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        print("STDERR:", r.stderr[-3000:])
        raise RuntimeError(f"Julia exited with code {r.returncode}")
    print(r.stdout[-5000:])
    return r.stdout

## 1. Per-component XLA traces

Each component gets its own Perfetto trace.

In [ ]:
jl(f'''
using Random, ConfParser, Lux, Reactant, Enzyme, ComponentArrays, Accessors, Statistics, JSON

include("src/utils.jl")
using .Utils

include("src/KAEM/KAEM.jl")
using .KAEM_model
using .KAEM_model.EBM_Model
using .KAEM_model.GeneratorModel

include("src/KAEM/model_setup.jl")
using .ModelSetup

include("src/pipeline/optimizer.jl")
include("src/pipeline/data_utils.jl")
using .optimization
using .DataUtils: get_vision_dataset

include("src/KAEM/rng.jl")
using .HLOrng

ENV["PERCEPTUAL"] = "false"
ENV["THERMO"] = "false"

trace_dir = "{TRACE_DIR}"
report_dir = "{REPORT_DIR}"

rng = Random.MersenneTwister(1)
conf = ConfParse("config/nist_config.ini")
parse_conf!(conf)
ENV["DEVICE"] = retrieve(conf, "TRAINING", "device")

dataset, x_shape, _ = get_vision_dataset("MNIST", 1000, 100, 100; img_resize=(32,32), cnn=false)
model = init_KAEM(dataset, conf, x_shape; rng=rng)
x_batch, _ = iterate(model.train_loader)
x_batch = pu(x_batch)
opt = create_opt(conf)
model, opt_st, ps, sk, sl, sr = prep_model(model, x_batch, opt; rng=rng, MLIR=false)
sr = seed_rand(model; rng=rng)

ebm = model.prior
gen = model.lkhood

function profile_component(label, f, args...; n_trace=30, n_time=20)
    println("\\n--- $$label ---")
    compiled = Reactant.@compile f(args...)
    compiled(args...)

    mkpath(trace_dir * "/$$label")
    Reactant.Profiler.with_profiler(trace_dir * "/$$label") do
        for _ in 1:n_trace
            compiled(args...)
        end
    end

    times = [(@elapsed compiled(args...)) for _ in 1:n_time]
    med = median(times) * 1000
    println("  $$(round(med, digits=3)) ms")
    return Dict("median_ms" => med, "min_ms" => minimum(times)*1000, "max_ms" => maximum(times)*1000)
end

results = Dict()
z = pu(randn(Float32, ebm.q_size, ebm.p_size, model.batch_size))

# EBM energy function
ebm_fwd(z, p, s, l) = ebm(p, s, l, z)
results["ebm_forward"] = profile_component("ebm_forward", ebm_fwd, z, ps.ebm, sk.ebm, sl.ebm)

# Generator / decoder
gen_fwd(p, s, l, z) = gen.output_activation(first(gen.generator(p, s, l, z)))
results["gen_forward"] = profile_component("gen_forward", gen_fwd, ps.gen, sk.gen, sl.gen, z)

# Gauss-Legendre quadrature
quad_fwd(p, s, l, q) = ebm.quad(ebm, p, s, l, q)
results["quadrature"] = profile_component("quadrature", quad_fwd, ps.ebm, sk.ebm, sl.ebm, sk.quad)

# Inverse transform sampling
its_fwd(m, p, s, l, r) = m.sample_prior(m, p, s, l, r)
results["inv_transform_sampling"] = profile_component("inv_transform_sampling", its_fwd, model, ps, sk, sl, sr)

# Log-prior
z_samp = pu(Float32.(Array(first(model.sample_prior(model, ps, sk, sl, sr)))))
lp_fwd(z, p, s, l, q) = model.log_prior(z, ebm, p, s, l, q)
results["log_prior"] = profile_component("log_prior", lp_fwd, z_samp, ps.ebm, sk.ebm, sl.ebm, sk.quad)

# Full generation (sample -> decode)
results["generation"] = profile_component("generation", model, ps, sk, Lux.testmode(sl), sr; n_trace=50)

# Full compiled train step
results["full_train_step"] = profile_component("full_train_step",
    model.train_step, opt_st, ps, sk, sl, x_batch, 1, sr; n_trace=30, n_time=30)

open(report_dir * "/component_times.json", "w") do f
    write(f, JSON.json(results, 2))
end
println("\\nTraces in ", trace_dir)
''')

## 2. HLO IR per component

In [ ]:
jl(f'''
using Random, ConfParser, Lux, Reactant, Enzyme, ComponentArrays, Accessors

include("src/utils.jl")
using .Utils

include("src/KAEM/KAEM.jl")
using .KAEM_model
using .KAEM_model.EBM_Model
using .KAEM_model.GeneratorModel

include("src/KAEM/model_setup.jl")
using .ModelSetup

include("src/pipeline/optimizer.jl")
include("src/pipeline/data_utils.jl")
using .optimization
using .DataUtils: get_vision_dataset

include("src/KAEM/rng.jl")
using .HLOrng

ENV["PERCEPTUAL"] = "false"
ENV["THERMO"] = "false"

report_dir = "{REPORT_DIR}"
mkpath(report_dir * "/hlo")

rng = Random.MersenneTwister(1)
conf = ConfParse("config/nist_config.ini")
parse_conf!(conf)
ENV["DEVICE"] = retrieve(conf, "TRAINING", "device")

dataset, x_shape, _ = get_vision_dataset("MNIST", 1000, 100, 100; img_resize=(32,32), cnn=false)
model = init_KAEM(dataset, conf, x_shape; rng=rng)
x_batch, _ = iterate(model.train_loader)
x_batch = pu(x_batch)
opt = create_opt(conf)
model, opt_st, ps, sk, sl, sr = prep_model(model, x_batch, opt; rng=rng, MLIR=false)
sr = seed_rand(model; rng=rng)

ebm = model.prior
gen = model.lkhood
z = pu(randn(Float32, ebm.q_size, ebm.p_size, model.batch_size))

targets = [
    ("ebm_forward",            () -> @code_hlo ebm(ps.ebm, sk.ebm, sl.ebm, z)),
    ("gen_forward",            () -> @code_hlo gen.generator(ps.gen, sk.gen, sl.gen, z)),
    ("quadrature",             () -> @code_hlo ebm.quad(ebm, ps.ebm, sk.ebm, sl.ebm, sk.quad)),
    ("inv_transform_sampling", () -> @code_hlo model.sample_prior(model, ps, sk, sl, sr)),
    ("log_prior",              () -> @code_hlo model.log_prior(z, ebm, ps.ebm, sk.ebm, sl.ebm, sk.quad)),
    ("generation",             () -> @code_hlo model(ps, sk, Lux.testmode(sl), sr)),
    ("train_step",             () -> @code_hlo model.train_step(opt_st, ps, sk, sl, x_batch, 1, sr)),
]

for (label, get_hlo) in targets
    print("$$label... ")
    try
        hlo_str = string(get_hlo())
        open(report_dir * "/hlo/$$label.txt", "w") do f
            write(f, hlo_str)
        end
        println("ok")
    catch e
        println("FAILED: ", e)
    end
end
println("HLO saved to ", report_dir * "/hlo/")
''')

## 3. Per-mode comparison (vanilla / variational / thermo)

In [ ]:
jl(f'''
using Random, ConfParser, Lux, Reactant, Enzyme, ComponentArrays, Accessors, Statistics, JSON

include("src/pipeline/trainer.jl")
using .trainer

include("src/KAEM/rng.jl")
using .HLOrng

ENV["PERCEPTUAL"] = "false"
trace_dir = "{TRACE_DIR}"
report_dir = "{REPORT_DIR}"
rng = Random.MersenneTwister(1)

modes = [
    ("vanilla",      "false", "0",  "false", "false"),
    ("variational",  "true",  "0",  "false", "false"),
    ("thermo",       "false", "4",  "true",  "true"),
]

summary = Dict()

for (label, variational, n_t, use_thermo, use_langevin) in modes
    println("\\n=== $$label ===")
    try
        conf = ConfParse("config/nist_config.ini")
        parse_conf!(conf)
        ENV["DEVICE"] = retrieve(conf, "TRAINING", "device")
        ENV["THERMO"] = use_thermo
        commit!(conf, "VARIATIONAL", "use_variational", variational)
        commit!(conf, "THERMODYNAMIC_INTEGRATION", "num_temps", n_t)
        commit!(conf, "POST_LANGEVIN", "use_langevin", use_langevin)

        compile_t = @elapsed begin
            t = init_trainer(rng, conf, "MNIST"; img_resize=(32,32), save_model=false)
        end

        t.st_rng = seed_rand(t.model; rng=t.rng)
        t.model.train_step(t.opt_state, t.ps, t.st_kan, t.st_lux, t.x, 1, t.st_rng)

        mkpath(trace_dir * "/mode_$$label")
        Reactant.Profiler.with_profiler(trace_dir * "/mode_$$label") do
            for _ in 1:30
                t.st_rng = seed_rand(t.model; rng=t.rng)
                t.model.train_step(t.opt_state, t.ps, t.st_kan, t.st_lux, t.x, 1, t.st_rng)
            end
        end

        times = Float64[]
        for _ in 1:20
            t.st_rng = seed_rand(t.model; rng=t.rng)
            push!(times, @elapsed t.model.train_step(t.opt_state, t.ps, t.st_kan, t.st_lux, t.x, 1, t.st_rng))
        end

        summary[label] = Dict(
            "compile_s" => compile_t,
            "median_ms" => median(times) * 1000,
            "min_ms" => minimum(times) * 1000,
            "max_ms" => maximum(times) * 1000,
        )
        println("  compile: $$(round(compile_t, digits=1))s, median: $$(round(median(times)*1000, digits=2))ms")
    catch e
        println("  FAILED: ", e)
        summary[label] = Dict("error" => string(e))
    end
end

open(report_dir * "/mode_summary.json", "w") do f
    write(f, JSON.json(summary, 2))
end
println("\\nSaved to ", report_dir)
''')

## 4. Figures

In [ ]:
import matplotlib.pyplot as plt

with open(f"{REPORT_DIR}/component_times.json") as f:
    comp = json.load(f)

labels = sorted(comp.keys(), key=lambda k: comp[k]["median_ms"], reverse=True)
times = [comp[k]["median_ms"] for k in labels]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels, times)
ax.set_xlabel("ms")
ax.set_title(f"KAEM component times ({DEVICE_NAME})")
ax.invert_yaxis()
for i, t in enumerate(times):
    ax.text(t + max(times)*0.01, i, f"{t:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/component_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

with open(f"{REPORT_DIR}/mode_summary.json") as f:
    modes = json.load(f)

ok = [m for m in ["vanilla", "variational", "thermo"] if m in modes and "error" not in modes[m]]
if ok:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.bar(ok, [modes[m]["median_ms"] for m in ok])
    a1.set_ylabel("ms/step")
    a1.set_title(f"Train step ({DEVICE_NAME})")
    a2.bar(ok, [modes[m]["compile_s"] for m in ok])
    a2.set_ylabel("seconds")
    a2.set_title(f"XLA compile ({DEVICE_NAME})")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/mode_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. List artifacts

In [ ]:
import glob
for p in sorted(glob.glob(f"{REPORT_DIR}/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"  {os.path.relpath(p, KAEM)}  ({os.path.getsize(p)//1024}KB)")
for p in sorted(glob.glob(f"{FIG_DIR}/*")):
    if os.path.isfile(p):
        print(f"  {os.path.relpath(p, KAEM)}")
print(f"\nOpen .pb traces at https://ui.perfetto.dev")

## 6. Push results to fork

Run this before the VM dies. Uses a Colab secret (add `GITHUB_TOKEN` with a [personal access token](https://github.com/settings/tokens); repo scope).

In [ ]:
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")

!cd /content/KAEM && git remote set-url origin https://{token}@github.com/PritRaj1/KAEM.git
!cd /content/KAEM && git config user.email "prithvi@zscc.ai"
!cd /content/KAEM && git config user.name "PritRaj1"
!cd /content/KAEM && git add profiling_reports/{DEVICE}/ figures/profiling/{DEVICE}/
!cd /content/KAEM && git commit -m "profiling results: {DEVICE}"
!cd /content/KAEM && git push